# V2D Challenge, Track 3: what we ran on Google Colab (explained)

**Short version.** NVIDIA's *Video-to-Data (V2D) Challenge, Track 3* gives you head-camera videos of a person handling everyday objects (cups, a pot, a dish rack, a dustpan...). You must make a **simulated pair of robot hands** reproduce what happened to the objects. The leaderboard scores how close the **simulated** object motion is to the real motion, which was recorded with motion capture (mocap).

So far on Colab we have:
1. Rented a strong GPU (NVIDIA RTX PRO 6000, 98 GB) and installed NVIDIA's robot simulator. We checked that it trains and evaluates.
2. Built our own **perception pipeline**, which recovers the 3-D position and rotation of every object in every video frame from the stereo cameras alone.
3. Measured it on all 20 public episodes, which come with mocap ground truth: **mean AUC 0.809** from video alone. For scale, the current leaderboard leader scores **0.765** after simulation, and a robot that does nothing scores about **0.446**.

The robot part (training a policy per video) is the next step and is not in this notebook yet.

> **How to read this notebook.** Text cells explain, code cells do. Section 4 is runnable end to end on a Colab **G4** GPU (Runtime → Change runtime type → G4). It re-creates our result on public episode 12. The other sections document what we ran and what we saw.

## 0. The challenge and how it is scored

**Input per episode.** About 7-18 s of video from a head-mounted camera rig with three cameras:
- `ego_cam_a`: a colour camera, 2028×1520.
- `ego_cam_b` and `ego_cam_c`: two black-and-white cameras 7.5 cm apart, a *stereo pair*. Like two eyes, they let us measure depth.

We also get 3-D scans (meshes) of the objects, except the wooden spoon, which has no scan.

**What must come out.** For each episode, a trained robot **policy**: a neural-network controller for two simulated *Sharpa* robot hands in NVIDIA's **FlashCHORD** simulator. The organisers' evaluator runs the policy once, records where the simulated objects go, and compares that with mocap.

**The pipeline we are building:**
```
video ──► (1) PERCEPTION: camera + every object's 3-D pose per frame   ◄── this notebook
      ──► (2) HANDS: where the person's hands and fingers are
      ──► (3) REFERENCE: turn (1)+(2) into a target motion for the robot hands
      ──► (4) TRAIN a policy in the simulator (reinforcement learning, ~1.5-2 h per video)
      ──► (5) OFFICIAL EVALUATOR replays the policy ──► simulated object trajectories
      ──► (6) SCORE vs mocap (leaderboard)
```

**Metrics.** The scorer first lines up one object's first pose (one rigid alignment) and never fixes scale. Then:

| metric | meaning | better |
|---|---|---|
| **ADD** | average distance between predicted and true object surface points (cm) | lower |
| **AUC** (main ranking) | for thresholds 1…9 cm, the share of frames with ADD below the threshold, averaged; 0 to 1 | higher |
| **SP-SR** | success if the average position error is ≤ 10 cm and the average angle error ≤ 0.5 rad | higher |
| **MP-SR** | strict success: **every** object within 3 cm and 30° on average | higher |
| **MPPE** | mean error of 6 points placed 5 cm around each object (cm) | lower |

Reference points: leaderboard leader **0.765 AUC** (12 Oct 2026). Every other team is between 0.23 and 0.37. A robot that leaves everything where it started scores about **0.446**.

## 1. What machine did we get?
This cell prints the GPU, CUDA version, OS, CPU and RAM, and checks for Docker.

**What we saw.** The free tier gave a Tesla **T4** (16 GB), too small. With Colab Pro we switched to **G4 = NVIDIA RTX PRO 6000 Blackwell, 98 GB**, plus 48 CPUs and 176 GB RAM, on Ubuntu 24.04 with driver 580 / CUDA 13.0 and Python 3.13. G4 costs about **8.9 compute units per hour** (a T4 is about 1.1).

**No Docker.** NVIDIA's own reconstruction pipeline runs as Docker containers, so on Colab we install the pieces we need one by one instead. Colab machines are also **not persistent**: when the runtime is recycled, all installs are lost. That is why section 4 has a one-shot `setup.sh`.

In [ ]:
!nvidia-smi
!nvcc --version || true
!cat /etc/os-release | head -2; python --version; nproc; free -g | head -2; df -h / | tail -1
!which docker || echo 'no docker'

## 2. NVIDIA's code and the robot simulator
- We download NVIDIA's `video_to_data` repository at **the exact commit the official evaluator uses** (`8c778f90…`). Scores must come from that version.
- Large files (robot-hand meshes, example data) are stored with **Git LFS**, so we pull only the folders we need (about 400 MB).
- The simulator (**FlashCHORD**: JAX + NVIDIA Warp + Newton + MuJoCo-Warp) goes into its own Python 3.11 environment, created with `uv`.

**What we saw.** Everything installed. JAX sees the GPU (`CudaDevice(id=0)`), with warp 1.14.0, newton 1.2.0 and mujoco 3.8.1. This takes about 5 minutes.

In [ ]:
%cd /content
!apt-get -qq install -y git-lfs > /dev/null
!rm -rf v2d && git init -q v2d && cd v2d && git remote add origin https://github.com/nvidia-isaac/video_to_data.git && GIT_LFS_SKIP_SMUDGE=1 git fetch -q --depth 1 origin 8c778f905cb35f76f8e6cd6e2ea486da4fb9e0ef && GIT_LFS_SKIP_SMUDGE=1 git checkout -q FETCH_HEAD && git log --oneline -1
%cd /content/v2d
!git lfs pull --include 'robotic_grounding/flash_chord/src/flash_chord/assets/**,robotic_grounding/source/robotic_grounding/robotic_grounding/assets/human_motion_data/ego_recon/**'
!pip -q install uv
%cd /content/v2d/robotic_grounding/flash_chord
!uv python install 3.11 && uv venv -q --python 3.11 .venv && UV_LINK_MODE=copy uv sync --frozen --extra train 2>&1 | tail -3
!.venv/bin/python -c 'import jax, warp, newton, mujoco; print(jax.__version__, jax.devices()); print(warp.__version__, newton.__version__, mujoco.__version__)'

## 3. Simulator smoke test: can we train a robot policy here, and how fast?
We train on NVIDIA's own example (a person moving a tissue box) for 20 million simulation steps, about 8% of a full training run.

- **Reinforcement learning (RL):** the policy controls the two robot hands. It is rewarded for making the simulated object follow the target motion. **4,096 copies of the scene** run in parallel on the GPU.
- A full policy is **250 million steps**.

**What we saw.**
- **~45,300 simulation steps per second**, **40.5 GB** of GPU memory, 100% GPU use.
- 20M steps took 7.4 minutes, so **one full policy takes about 1.5-2 hours (about 15-18 Colab units)**.
- We also ran the **official evaluator** on that policy (1 world, trajectory export). It works. Against the policy's own target it scored AUC 0.918, MP-SR 1.0, MPPE 1.63 cm.

In [ ]:
%cd /content/v2d/robotic_grounding/flash_chord
!export A=/content/v2d/robotic_grounding/source/robotic_grounding/robotic_grounding/assets P=/content/v2d/robotic_grounding/source/robotic_grounding/robotic_grounding/assets/human_motion_data/ego_recon/processed/sequence_id=tissue_box_simple/robot_name=sharpa_wave && export V2D_ASSETS_DIR=$A XLA_PYTHON_CLIENT_PREALLOCATE=false MUJOCO_GL=egl && .venv/bin/python scripts/train_flash_sac.py experiment=sharpa_flash_sac task.parquet=\'$P\' reset.seed=42 logging.mode=disabled output_dir=outputs/smoke training.total_environment_steps=20004864 2>&1 | grep -E 'progress|completed' | tail -3

Running the **official evaluator** on that policy (one simulated world, exporting the object trajectories):

In [ ]:
%cd /content/v2d/robotic_grounding/flash_chord
!CK=$(ls -v outputs/smoke/policy_*.safetensors | tail -1); MUJOCO_GL=egl XLA_PYTHON_CLIENT_PREALLOCATE=false .venv/bin/python scripts/evaluate_policy.py evaluation.checkpoint=$CK evaluation.world_count=1 evaluation.reset_mode=explicit evaluation.metrics_output=/content/smoke_eval.json evaluation.object_trajectories_output=/content/smoke_traj.parquet hydra.run.dir=/content/smoke_eval logging.mode=disabled 2>&1 | tail -3; cat /content/smoke_eval.json | head -40

## 4. Our perception pipeline: from stereo video to 3-D object motion
This is the part we built and measured. For every frame we want each object's **6-D pose**: 3-D position plus 3-D rotation, in one fixed world frame. The pipeline:

1. **Rectify the stereo pair.** We warp the two black-and-white cameras so that matching points sit on the same image row. That makes depth estimation a 1-D search.
2. **Depth from stereo:** NVIDIA's **FoundationStereo** network gives a depth map per frame. Depth = focal length × baseline / disparity (focal ≈ 600 px, baseline 7.5 cm). This takes about 0.27 s per frame.
3. **Find each object in the first frame** (section 5 explains why we don't use text prompts):
   - **SAM2** (Meta) cuts the image into candidate regions.
   - We render each object's 3-D scan from 42 viewpoints in grey, to match the black-and-white cameras.
   - **DINOv2** (Meta) image features compare every region with those renders.
   - A **3-D size check** from the depth rejects regions that are too big or too small.
   - Objects are matched to regions with a one-to-one assignment (Hungarian algorithm). Brightness breaks ties between twin objects, e.g. a dark blue cup versus a light beige one.
4. **Track each object:** NVIDIA's **FoundationPose** registers the scan to the first frame, then tracks it frame by frame using image + depth. It runs at about 20 fps.
5. **Track the camera:** the head moves. We match image features (SIFT) to earlier frames, lift them to 3-D with the depth, and solve for the camera pose (PnP with RANSAC). This is called *visual odometry*.
6. **Combine:** object pose in the world = camera pose × object pose seen by the camera.
7. **Score.** Only now do we read the mocap ground truth, using the official kit's metric code. The pipeline itself never sees ground truth.

**Cost.** About 1-3 minutes per episode on the G4 after setup. Setup (next cell) takes about 15 minutes.

### 4a. One-shot setup
This installs everything perception needs and downloads public episode 12 plus the official scoring kit. Each step prints a marker such as `SAM2-OK`. It also installs the simulator environment from section 2 in the background.

In [ ]:
%%writefile /content/setup.sh
#!/bin/bash
# One-shot Colab (G4 / RTX PRO 6000 Blackwell) setup for V2D Track 3 experiments.
# Usage: bash /content/setup.sh > /content/setup.log 2>&1 &   (then: grep -E 'OK|DONE|rror' /content/setup.log)
cd /content
export CUDA_HOME=/usr/local/cuda TORCH_CUDA_ARCH_LIST=12.0 MAX_JOBS=40
apt-get -qq install -y git-lfs libeigen3-dev libboost-all-dev > /dev/null 2>&1

# --- FlashCHORD (NVIDIA evaluator pin) in its own uv venv, in the background ---
(
  if [ ! -d v2d ]; then
    git init -q v2d && cd v2d && git remote add origin https://github.com/nvidia-isaac/video_to_data.git &&
    GIT_LFS_SKIP_SMUDGE=1 git fetch -q --depth 1 origin 8c778f905cb35f76f8e6cd6e2ea486da4fb9e0ef &&
    GIT_LFS_SKIP_SMUDGE=1 git checkout -q FETCH_HEAD &&
    git lfs pull --include 'robotic_grounding/flash_chord/src/flash_chord/assets/**,robotic_grounding/source/robotic_grounding/robotic_grounding/assets/human_motion_data/ego_recon/**'
    cd /content
  fi
  pip install -q uv
  cd /content/v2d/robotic_grounding/flash_chord && uv python install 3.11 > /dev/null &&
  uv venv -q --python 3.11 .venv && UV_LINK_MODE=copy uv sync -q --frozen --extra train && echo FLASHCHORD-OK
) > /content/setup_flashchord.log 2>&1 &

# --- Track 3 public episode data ---
EP=${EP:-12}
B=https://huggingface.co/datasets/nvidia/video_to_data_challenge/resolve/main/track_3/public
mkdir -p /content/t3/v /content/t3/mesh
for c in a b c; do f=/content/t3/v/ep${EP}_$c.mp4; [ -s $f ] || curl -sSL -o $f $B/videos/chunk-000/observation.images.ego_cam_$c/episode_$(printf %06d $EP).mp4; done
[ -s /content/t3/ep${EP}.parquet ] || curl -sSL -o /content/t3/ep${EP}.parquet $B/data/chunk-000/episode_$(printf %06d $EP).parquet
[ -s /content/t3/calib.json ] || curl -sSL -o /content/t3/calib.json $B/meta/camera_calibration.json
for o in blue_cup plastic_dish_rack; do mkdir -p /content/t3/mesh/$o; f=/content/t3/mesh/$o/$o.glb; [ -s $f ] || curl -sSL -o $f $B/mesh/$o/$o.glb; done
echo DATA-OK

# --- submission kit (metric code) ---
mkdir -p /content/kit && cd /content/kit && [ -d v2d_submission_kit ] || (curl -sSL -o kit.zip https://nvidia-isaac.github.io/video_to_data/v2d_challenge/assets/v2d_submission_kit.zip && unzip -q -o kit.zip) ; cd /content && echo KIT-OK

# --- perception python deps (system Python 3.13, torch 2.11+cu130) ---
pip install -q open3d trimesh omegaconf timm einops imageio gdown scikit-image ruamel.yaml imgaug albumentations joblib h5py transformations pyrender PyOpenGL kornia psutil pybind11 ninja pyarrow warp-lang && echo PYDEPS-OK

# SAM2
SAM2_BUILD_CUDA=0 pip install -q 'git+https://github.com/facebookresearch/sam2.git' && echo SAM2-OK
mkdir -p /content/ckpt && [ -s /content/ckpt/sam2.1_hiera_large.pt ] || curl -sSL -o /content/ckpt/sam2.1_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

# FoundationStereo (ViT-L 23-51-11)
[ -d FoundationStereo ] || git clone -q https://github.com/NVlabs/FoundationStereo.git
[ -d FoundationStereo/pretrained_models/23-51-11 ] || (cd FoundationStereo && gdown -q --folder https://drive.google.com/drive/folders/1VhPebc_mMxWKccrv7pdQLTvXYVcLYpsf -O pretrained_models)
ls FoundationStereo/pretrained_models/23-51-11 && echo FSTEREO-OK

# FoundationPose
[ -d FoundationPose ] || git clone -q https://github.com/NVlabs/FoundationPose.git
pip install -q --no-build-isolation 'git+https://github.com/NVlabs/nvdiffrast.git' && echo NVDIFFRAST-OK
pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/pytorch3d.git' && echo PYTORCH3D-OK
(cd FoundationPose/mycpp && rm -rf build && mkdir build && cd build &&
 cmake .. -DPYTHON_EXECUTABLE=$(which python) -Dpybind11_DIR=$(python -m pybind11 --cmakedir) > /dev/null && make -j40 > /dev/null) && echo MYCPP-OK
[ -d FoundationPose/weights/2023-10-28-18-33-37 ] || (cd FoundationPose && gdown -q --folder https://drive.google.com/drive/folders/1DFezOAD0oD1BblsXVxqDsl8fj0qzB82i -O weights)
ls FoundationPose/weights && echo FPOSE-OK

wait
echo SETUP-DONE

In [ ]:
!bash /content/setup.sh > /content/setup.log 2>&1; grep -aE '[A-Z0-9]+-OK|DONE' /content/setup.log

### 4b. The perception script
`perc.py` implements steps 1-7 above. It writes `result.json` (the scores), `poses.npz` (all poses) and `masks_f0.jpg` (which region was picked for each object) to `/content/perc/ep<N>/`.

It scores several *world* variants:
- `vo`: camera from visual odometry. This is the real method.
- `anchor:<object>`: treats one object as static and uses it as the world. This is for analysis only.

In [ ]:
%%writefile /content/perc.py
"""Perception-only accuracy on a public Track 3 episode (any objects).

Inference (no ground-truth poses read before scoring; object names/slot order come from the episode,
as the kit provides them for eval episodes):
  mono stereo pair c/b -> rectify -> FoundationStereo depth (rectified-left frame, cached)
  frame 0: OWLv2 candidate boxes per object prompt -> SAM2 masks -> FoundationPose register per
           (object, candidate); assignment by FoundationPose score (Hungarian within a prompt group)
  FoundationPose track_one on later frames
  camera: keyframe SIFT + depth PnP (world = rectified-left camera at frame 0)
Scoring: official kit Track 3 metric functions vs mocap, for world variants
  'vo' (camera from PnP) and 'anchor:<obj>' (that object assumed static; analysis only).
Env: EP, FS_SCALE (0.5), OUT. Outputs result.json, poses.npz, masks_f0.jpg, vo.json in OUT.
"""
import os, sys, json, time, importlib
import numpy as np, cv2, torch, trimesh
import pyarrow.parquet as pq
from scipy.spatial.transform import Rotation as Rot
from scipy.optimize import linear_sum_assignment

os.environ.setdefault('TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD', '1')
EP = int(os.environ.get('EP', 12))
FS_SCALE = float(os.environ.get('FS_SCALE', 0.5))
D = '/content/t3'
OUT = os.environ.get('OUT', f'/content/perc/ep{EP}')
os.makedirs(OUT, exist_ok=True)
PROMPT = {'white_pot': 'a cooking pot', 'white_pot_lid': 'a pot lid', 'blue_cup': 'a cup', 'beige_cup': 'a cup',
          'plastic_dish_rack': 'a dish rack', 'water_pitcher': 'a pitcher', 'mini_sweeper': 'a hand broom',
          'mini_dust_pan': 'a dustpan', 'wooden_piece_1': 'a wooden block', 'wooden_piece_2': 'a wooden block',
          'wooden_spoon': 'a wooden spoon'}
t_all = time.time()
def log(*a):
    print(f'[{time.time() - t_all:7.1f}s]', *a, flush=True)
def mesh_path(n):
    p = f'{D}/mesh/{n}/{n}.glb'
    return p if os.path.exists(p) else f'{D}/mesh/{n}/{n}_visual.glb'

tab = pq.read_table(f'{D}/ep{EP}.parquet').to_pandas()
names = [o['name'] for o in tab['observation.objects'].iloc[0]]          # slot order only
log('episode', EP, 'objects', names)

# ---------- calibration + rectification (left = ego_cam_c, right = ego_cam_b) ----------
C = json.load(open(f'{D}/calib.json')); cam = C['cameras']
sp = [p for p in C['stereo_pairs'] if p['camera'] == 'ego'][0]
K_ = lambda n: np.array(cam[n]['intrinsic_matrix'], float)
D_ = lambda n: np.array(cam[n]['distortion_coefficients'], float)
Kc, dc, Kb, db = K_('ego_cam_c'), D_('ego_cam_c'), K_('ego_cam_b'), D_('ego_cam_b')
E = np.array(sp['left_to_right'], float); R, T = E[:3, :3].copy(), E[:3, 3].reshape(3, 1).copy()
SZ, RSZ = (1280, 800), (1600, 1000)
R1, R2, P1, P2, Q, _, _ = cv2.stereoRectify(Kc, dc, Kb, db, SZ, R, T, alpha=0, newImageSize=RSZ, flags=cv2.CALIB_ZERO_DISPARITY)
MC = cv2.initUndistortRectifyMap(Kc, dc, R1, P1, RSZ, cv2.CV_32FC1)
MB = cv2.initUndistortRectifyMap(Kb, db, R2, P2, RSZ, cv2.CV_32FC1)
FR = P1[0, 0]; BASE = -P2[0, 3] / P2[0, 0]; KR = P1[:3, :3].copy()

def frames(path):
    cap = cv2.VideoCapture(path); out = []
    while True:
        ok, f = cap.read()
        if not ok: break
        out.append(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY))
    return out
fc, fb = frames(f'{D}/v/ep{EP}_c.mp4'), frames(f'{D}/v/ep{EP}_b.mp4')
N = min(len(fc), len(fb), len(tab))
rc = [cv2.remap(fc[i], MC[0], MC[1], cv2.INTER_LINEAR) for i in range(N)]
rb = [cv2.remap(fb[i], MB[0], MB[1], cv2.INTER_LINEAR) for i in range(N)]
rgb = lambda g: np.repeat(g[..., None], 3, 2)
log('frames c/b/rows', len(fc), len(fb), len(tab), 'focal', round(FR, 1), 'baseline', round(BASE, 4))

# ---------- FoundationStereo depth (cached) ----------
DCACHE = f'{OUT}/depths_fs{FS_SCALE}.npy'
if os.path.exists(DCACHE):
    depths = list(np.load(DCACHE))
else:
    sys.path.insert(0, '/content/FoundationStereo')
    from omegaconf import OmegaConf
    from core.utils.utils import InputPadder
    from core.foundation_stereo import FoundationStereo
    CK = '/content/FoundationStereo/pretrained_models/23-51-11/model_best_bp2.pth'
    cfg = OmegaConf.load(os.path.join(os.path.dirname(CK), 'cfg.yaml'))
    cfg['valid_iters'] = 32; cfg['hiera'] = 0
    cfg['vit_size'] = cfg.get('vit_size', 'vitl')
    fs = FoundationStereo(cfg); fs.load_state_dict(torch.load(CK, weights_only=False)['model']); fs.cuda().eval()
    depths = []
    with torch.no_grad():
        for i in range(N):
            a = cv2.resize(rc[i], None, fx=FS_SCALE, fy=FS_SCALE); b = cv2.resize(rb[i], None, fx=FS_SCALE, fy=FS_SCALE)
            t0 = torch.as_tensor(rgb(a)).cuda().float()[None].permute(0, 3, 1, 2)
            t1 = torch.as_tensor(rgb(b)).cuda().float()[None].permute(0, 3, 1, 2)
            pad = InputPadder(t0.shape, divis_by=32, force_square=False); t0, t1 = pad.pad(t0, t1)
            with torch.amp.autocast('cuda', enabled=True):
                disp = fs.forward(t0, t1, iters=32, test_mode=True)
            disp = pad.unpad(disp.float()).cpu().numpy().reshape(a.shape[0], a.shape[1])
            disp = cv2.resize(disp, RSZ, interpolation=cv2.INTER_LINEAR) / FS_SCALE
            z = np.where(disp > 1, FR * BASE / np.maximum(disp, 1e-6), 0).astype(np.float32)
            z[(z < 0.15) | (z > 2.0)] = 0
            depths.append(z)
    np.save(DCACHE, np.stack(depths).astype(np.float32)); del fs; torch.cuda.empty_cache()
    for k in [k for k, m in list(sys.modules.items()) if (getattr(m, '__file__', None) or '').startswith('/content/FoundationStereo')]:
        del sys.modules[k]
    sys.path.remove('/content/FoundationStereo')
log('depth ready', len(depths))

# ---------- camera: keyframe SIFT + depth PnP ----------
sift = cv2.SIFT_create(nfeatures=4000); bf = cv2.BFMatcher(cv2.NORM_L2)
F = [sift.detectAndCompute(rc[i], None) for i in range(N)]
fx, cx, cy = KR[0, 0], KR[0, 2], KR[1, 2]
def pnp(kf, i):
    kpK, dK = F[kf]; kpi, di = F[i]
    if dK is None or di is None: return None, 0
    good = [p[0] for p in bf.knnMatch(dK, di, k=2) if len(p) == 2 and p[0].distance < 0.75 * p[1].distance]
    P3, P2_ = [], []
    for g in good:
        u, v = kpK[g.queryIdx].pt; z = depths[kf][int(round(v)), int(round(u))]
        if z <= 0: continue
        P3.append([(u - cx) * z / fx, (v - cy) * z / fx, z]); P2_.append(kpi[g.trainIdx].pt)
    if len(P3) < 30: return None, 0
    P3, P2_ = np.float32(P3), np.float32(P2_)
    ok, rv, tv, inl = cv2.solvePnPRansac(P3, P2_, KR, None, iterationsCount=2000, reprojectionError=1.5,
                                         confidence=0.999, flags=cv2.SOLVEPNP_SQPNP)
    if not ok or inl is None or len(inl) < 20: return None, 0
    inl = inl[:, 0]
    rv, tv = cv2.solvePnPRefineLM(P3[inl], P2_[inl], KR, None, rv, tv)
    M = np.eye(4); M[:3, :3] = cv2.Rodrigues(rv)[0]; M[:3, 3] = tv[:, 0]
    return M, len(inl)                                                     # cam_i <- cam_kf
KF_MIN = int(os.environ.get('KF_MIN', 250))
W_C = [np.eye(4)]; kf = 0; kfs = [0]; ninl = [0]; fails = 0
for i in range(1, N):
    M, n = pnp(kf, i)
    if n < KF_MIN and i - 1 > kf:
        kf = i - 1; kfs.append(kf); M, n = pnp(kf, i)
    if M is None:
        fails += 1; W_C.append(W_C[-1].copy()); ninl.append(0); continue
    W_C.append(W_C[kf] @ np.linalg.inv(M)); ninl.append(n)
W_C = np.stack(W_C)
log('vo keyframes', len(kfs), 'fails', fails, 'median inliers', int(np.median(ninl[1:])), 'min', int(np.min(ninl[1:])))
json.dump({'keyframes': kfs, 'inliers': ninl, 'fails': fails}, open(f'{OUT}/vo.json', 'w'))

# ---------- FoundationPose setup (gray textures: the ego stereo cameras are monochrome) ----------
os.chdir('/content/FoundationPose'); sys.path.insert(0, '/content/FoundationPose')
from estimater import FoundationPose, ScorePredictor, PoseRefinePredictor
from Utils import nvdiffrast_render, make_mesh_tensors
import nvdiffrast.torch as dr
scorer, refiner, glctx = ScorePredictor(), PoseRefinePredictor(), dr.RasterizeCudaContext()
def load_mesh(n):
    m = trimesh.load(mesh_path(n), force='mesh')
    try: m.visual = m.visual.to_color()
    except Exception as e: log('to_color failed', n, e)
    if os.environ.get('GRAY', '1') == '1':
        vc = np.asarray(m.visual.vertex_colors).copy()
        vc[:, :3] = (0.299 * vc[:, 0] + 0.587 * vc[:, 1] + 0.114 * vc[:, 2]).astype(np.uint8)[:, None]
        m.visual.vertex_colors = vc
    return m
meshes = {n: load_mesh(n) for n in names}

# ---------- frame-0 detection: SAM2 proposals x DINOv2 vs gray mesh renders (CNOS-style) + 3-D size gate ----------
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
gen = SAM2AutomaticMaskGenerator(build_sam2('configs/sam2.1/sam2.1_hiera_l.yaml', '/content/ckpt/sam2.1_hiera_large.pt', device='cuda'),
                                 points_per_side=32, pred_iou_thresh=0.7, stability_score_thresh=0.85)
img0 = rgb(rc[0]); props = [m['segmentation'] for m in gen.generate(img0) if m['area'] >= 400]
del gen; torch.cuda.empty_cache()
def extent3d(m):
    v, u = np.nonzero(m & (depths[0] > 0))
    if len(v) < 30: return 0.0
    z = depths[0][v, u]; X = np.stack([(u - cx) * z / fx, (v - cy) * z / fx, z], 1)
    X = X[::max(1, len(X) // 5000)]; X = X - np.median(X, 0)
    pr = X @ np.linalg.svd(X, full_matrices=False)[2].T
    return float((np.percentile(pr, 97, 0) - np.percentile(pr, 3, 0)).max())
def crop(img, m):
    v, u = np.nonzero(m); y0, y1, x0, x1 = v.min(), v.max() + 1, u.min(), u.max() + 1
    s = int(max(y1 - y0, x1 - x0) * 1.1) + 2; ys, xs = (y0 + y1) // 2 - s // 2, (x0 + x1) // 2 - s // 2
    im = np.where(m[..., None], img, 0).astype(np.uint8); pad = np.zeros((s, s, 3), np.uint8)
    a0, b0, a1, b1 = max(ys, 0), max(xs, 0), min(ys + s, img.shape[0]), min(xs + s, img.shape[1])
    pad[a0 - ys:a1 - ys, b0 - xs:b1 - xs] = im[a0:a1, b0:b1]
    return cv2.resize(pad, (224, 224), interpolation=cv2.INTER_AREA)
dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14').cuda().eval()
MEAN = torch.tensor([0.485, 0.456, 0.406]).cuda()[:, None, None]; STD = torch.tensor([0.229, 0.224, 0.225]).cuda()[:, None, None]
def desc(crops):
    out = []
    with torch.no_grad():
        for k in range(0, len(crops), 64):
            x = (torch.as_tensor(np.stack(crops[k:k + 64])).cuda().float().permute(0, 3, 1, 2) / 255 - MEAN) / STD
            out.append(torch.nn.functional.normalize(dino(x), dim=-1))
    return torch.cat(out)
def templates(n):
    m = meshes[n]; c = m.bounds.mean(0); r = float(np.linalg.norm(m.extents)) / 2; dist = 4 * r
    Kt = np.array([[0.8 * 112 * dist / r, 0, 112], [0, 0.8 * 112 * dist / r, 112], [0, 0, 1]]); TP = []
    for d in trimesh.creation.icosphere(subdivisions=1).vertices:
        d = d / np.linalg.norm(d); z = -d; up = np.array([0, 0, 1.0]) if abs(z[2]) < 0.95 else np.array([0, 1.0, 0])
        x = np.cross(up, z); x /= np.linalg.norm(x); y = np.cross(z, x); Roc = np.stack([x, y, z], 1)
        M = np.eye(4); M[:3, :3] = Roc.T; M[:3, 3] = -Roc.T @ (c + d * dist); TP.append(M)
    col, dep, _ = nvdiffrast_render(K=Kt, H=224, W=224, ob_in_cams=torch.as_tensor(np.stack(TP)).float().cuda(), glctx=glctx,
                                    mesh_tensors=make_mesh_tensors(m), use_light=True)
    col = (col.clamp(0, 1) * 255).cpu().numpy().astype(np.uint8); msk = dep.cpu().numpy() > 0
    ks = [k for k in range(len(col)) if msk[k].sum() > 50]
    return desc([crop(col[k], msk[k]) for k in ks]), float(np.mean([col[k][msk[k]].mean() for k in ks]))
PD = desc([crop(img0, m) for m in props])
pint = np.array([img0[m].mean() for m in props]); pext = np.array([extent3d(m) for m in props])
S = np.full((len(names), len(props)), -10.0); LAM_I = float(os.environ.get('LAM_I', 0.3))
for a, n in enumerate(names):
    Tn, tint = templates(n); sim = (PD @ Tn.T).topk(min(5, len(Tn)), dim=1).values.mean(1).cpu().numpy()
    ext = float(meshes[n].bounding_box_oriented.extents.max()); ok = (pext > 0.3 * ext) & (pext < 1.3 * ext)
    S[a] = np.where(ok, sim - LAM_I * np.abs(pint - tint) / 255, -10.0)
    log('det', n, 'ext', round(ext, 3), 'tint', round(tint), [(int(k), round(S[a, k], 3), round(pext[k], 3), round(pint[k])) for k in np.argsort(-S[a])[:4]])
del dino; torch.cuda.empty_cache()
ri, ci = linear_sum_assignment(-S); MASK = {names[a]: props[k] for a, k in zip(ri, ci)}
DET = {names[a]: [int(k), float(S[a, k])] for a, k in zip(ri, ci)}; log('proposals', len(props), 'assignment', DET)
vis = img0.copy(); cols = [(0, 0, 255), (0, 255, 0), (255, 0, 0), (0, 255, 255)]
for j, n in enumerate(names):
    m = MASK[n]; vis[m] = (0.5 * vis[m] + 0.5 * np.array(cols[j % 4])).astype(np.uint8)
cv2.imwrite(f'{OUT}/masks_f0.jpg', vis)
ests = {n: FoundationPose(model_pts=meshes[n].vertices, model_normals=meshes[n].vertex_normals, mesh=meshes[n],
                          scorer=scorer, refiner=refiner, debug_dir=f'{OUT}/dbg', debug=0, glctx=glctx) for n in names}
poses = {}
for n in names:
    t0 = time.time()
    P = [ests[n].register(K=KR, rgb=rgb(rc[0]), depth=depths[0], ob_mask=MASK[n], iteration=5)]
    for i in range(1, N):
        P.append(ests[n].track_one(rgb=rgb(rc[i]), depth=depths[i], K=KR, iteration=2))
    poses[n] = np.stack(P); log('tracked', n, 'sec', round(time.time() - t0, 1))
np.savez(f'{OUT}/poses.npz', world_T_cam=W_C, **{f'cam_T_{n}': poses[n] for n in names})

# ---------- scoring (GT read only here) ----------
def T_of(p):  # [x,y,z,qw,qx,qy,qz] -> 4x4
    M = np.eye(4); M[:3, :3] = Rot.from_quat([p[4], p[5], p[6], p[3]]).as_matrix(); M[:3, 3] = p[:3]; return M
def xyzw(M):
    return np.r_[M[:3, 3], Rot.from_matrix(M[:3, :3]).as_quat()]
gt = {}
for row in tab['observation.objects'].iloc[:N]:
    for o in row: gt.setdefault(o['name'], []).append(T_of(np.array(o['pose'], float)))
gt = {k: np.stack(v) for k, v in gt.items()}
sys.path.insert(0, '/content/kit/v2d_submission_kit/metric_code/track_3')
A = importlib.import_module('AUC')
VERTS = {}
for n in names:
    pts, _ = trimesh.sample.sample_surface(trimesh.load(mesh_path(n), force='mesh'), 500, seed=0); VERTS[n] = np.asarray(pts)
def score(pred, objs):
    ach = np.stack([[xyzw(pred[n][i]) for n in objs] for i in range(N)])[:, None]
    ref = np.stack([[xyzw(gt[n][i]) for n in objs] for i in range(N)])[:, None]
    ach = A._warp_to_reference_start(ach, ref)
    add = A._object_add(ach, ref, np.stack([VERTS[n] for n in objs]))
    pos = np.linalg.norm(ach[..., :3] - ref[..., :3], axis=-1)[:, 0]
    return dict(AUC=float(A._add_auc(add)), SP_SR=float(np.mean(A._spider_success(ach, ref))),
                MP_SR=float(np.mean(A._maniptrans_success(ach, ref, np.arange(len(objs))))),
                MPPE_cm=float(A._mean_per_point_error_cm(ach, ref)), RPE_cm=float(A._relative_position_error_cm(ach, ref)),
                per_obj_AUC={n: float(A._add_auc(add[:, :, b:b + 1])) for b, n in enumerate(objs)},
                pos_err_cm_median={n: float(np.median(pos[:, b]) * 100) for b, n in enumerate(objs)},
                pos_err_cm_max={n: float(pos[:, b].max() * 100) for b, n in enumerate(objs)})
variants = {'vo': W_C}
for a in names: variants[f'anchor:{a}'] = np.stack([np.linalg.inv(M) for M in poses[a]])
res = {'episode': EP, 'objects': names, 'frames': N, 'fs_scale': FS_SCALE,
       'vo_stats': {'keyframes': len(kfs), 'fails': fails, 'median_inliers': int(np.median(ninl[1:]))},
       'det': DET, 'n_proposals': len(props),
       'gt_motion_extent_cm': {n: float(np.linalg.norm(gt[n][:, :3, 3] - gt[n][0, :3, 3], axis=1).max() * 100) for n in names}}
for v, WC in variants.items():
    pred = {n: np.stack([WC[i] @ poses[n][i] for i in range(N)]) for n in names}
    res[v] = score(pred, names)
    if v == 'vo': np.savez(f'{OUT}/poses_world_vo.npz', **pred)
json.dump(res, open(f'{OUT}/result.json', 'w'), indent=1)
for v in variants:
    r = res[v]; log(f'{v:28s} AUC {r["AUC"]:.3f} SP {r["SP_SR"]:.2f} MP {r["MP_SR"]:.2f} MPPE {r["MPPE_cm"]:.2f} RPE {r["RPE_cm"]:.2f}',
                    {n: round(x, 3) for n, x in r['per_obj_AUC'].items()})
log('DONE')

### 4c. Run it on public episode 12 (a blue cup put into a dish rack)
**What the log lines mean:**
- `rectified focal … baseline …` is the stereo geometry: about 600 px and 0.0755 m.
- `depth ready` means stereo depth is done for all frames. It is cached, so re-runs skip it.
- `vo keyframes … median inliers …` comes from camera tracking. More inliers is better; many keyframes means longer chains and more drift.
- `det <object> …` lists the best candidate regions with their scores and 3-D sizes, and `assignment` is the final pick.
- `tracked <object>` reports FoundationPose tracking time.
- The last lines are the **scores**. Look at the `vo` line.

**What we got on episode 12:** `vo AUC 0.995, SP 1.00, MP 1.00, MPPE 0.59 cm`, with a cup AUC of 1.00 and a rack AUC of 0.99. The cup moved 22 cm, and our median position error was about 1 cm.

In [ ]:
!cd /content && EP=12 python /content/perc.py 2>&1 | grep -a '^\['

In [ ]:
from IPython.display import Image, display
display(Image('/content/perc/ep12/masks_f0.jpg', width=800))   # first frame: picked regions (blue = slot 0 object, green = slot 1)

### 4d. Diagnostics: frame-by-frame error and overlay
`diag.py` prints the position and rotation error per object every N frames, after the same alignment the scorer uses. It also draws the tracked scan (**green dots**) and the mocap truth (**red dots**) on 8 frames. When tracking is good the two coincide. This is how we found the failures in section 5.

In [ ]:
%%writefile /content/diag.py
"""Diagnostics (evaluation only): per-frame pose error of tracked objects vs mocap after the scorer's
slot-0 frame-0 alignment, plus an overlay montage (pred = green, GT = red) on rectified-left frames."""
import sys, json, numpy as np, cv2, trimesh, pyarrow.parquet as pq
from scipy.spatial.transform import Rotation as Rot
EP = int(sys.argv[1]); ROOT = sys.argv[2] if len(sys.argv) > 2 else '/content/perc2'; STEP = int(sys.argv[3]) if len(sys.argv) > 3 else 20
D = '/content/t3'; O = f'{ROOT}/ep{EP}'
P = np.load(f'{O}/poses.npz'); W = P['world_T_cam']
tab = pq.read_table(f'{D}/ep{EP}.parquet').to_pandas(); names = [o['name'] for o in tab['observation.objects'].iloc[0]]
N = len(W)
def T_of(p):
    M = np.eye(4); M[:3, :3] = Rot.from_quat([p[4], p[5], p[6], p[3]]).as_matrix(); M[:3, 3] = p[:3]; return M
gt = {n: np.stack([T_of(np.array([o for o in row if o['name'] == n][0]['pose'], float)) for row in tab['observation.objects'].iloc[:N]]) for n in names}
pred = {n: np.einsum('tij,tjk->tik', W, P[f'cam_T_{n}']) for n in names}
Q = pred[names[0]][0] @ np.linalg.inv(gt[names[0]][0])           # GT -> our world (slot-0 frame-0 alignment)
gtw = {n: np.einsum('ij,tjk->tik', Q, gt[n]) for n in names}
print('frame | ' + ' | '.join(f'{n[:12]:12s} err_cm rot_deg gtmove_cm' for n in names))
for i in range(0, N, STEP):
    s = []
    for n in names:
        e = np.linalg.norm(pred[n][i][:3, 3] - gtw[n][i][:3, 3]) * 100
        r = np.degrees(np.linalg.norm(Rot.from_matrix(pred[n][i][:3, :3] @ gtw[n][i][:3, :3].T).as_rotvec()))
        mv = np.linalg.norm(gtw[n][i][:3, 3] - gtw[n][0][:3, 3]) * 100
        s.append(f'{n[:12]:12s} {e:6.1f} {r:7.1f} {mv:9.1f}')
    print(f'{i:5d} | ' + ' | '.join(s))
C = json.load(open(f'{D}/calib.json')); cam = C['cameras']; sp = [p for p in C['stereo_pairs'] if p['camera'] == 'ego'][0]
K_ = lambda n: np.array(cam[n]['intrinsic_matrix'], float); D_ = lambda n: np.array(cam[n]['distortion_coefficients'], float)
E = np.array(sp['left_to_right'], float)
R1, R2, P1, P2, _, _, _ = cv2.stereoRectify(K_('ego_cam_c'), D_('ego_cam_c'), K_('ego_cam_b'), D_('ego_cam_b'), (1280, 800), E[:3, :3], E[:3, 3:], alpha=0, newImageSize=(1600, 1000), flags=cv2.CALIB_ZERO_DISPARITY)
MC = cv2.initUndistortRectifyMap(K_('ego_cam_c'), D_('ego_cam_c'), R1, P1, (1600, 1000), cv2.CV_32FC1); KR = P1[:3, :3]
def mp(n):
    import os
    p = f'{D}/mesh/{n}/{n}.glb'; p = p if os.path.exists(p) else f'{D}/mesh/{n}/{n}_visual.glb'
    return np.asarray(trimesh.sample.sample_surface(trimesh.load(p, force='mesh'), 600, seed=0)[0])
V = {n: mp(n) for n in names}
cap = cv2.VideoCapture(f'{D}/v/ep{EP}_c.mp4'); fr = []
while True:
    ok, f = cap.read()
    if not ok: break
    fr.append(f)
idx = np.linspace(0, N - 1, 8).astype(int); tiles = []
for i in idx:
    img = cv2.cvtColor(cv2.remap(cv2.cvtColor(fr[i], cv2.COLOR_BGR2GRAY), MC[0], MC[1], cv2.INTER_LINEAR), cv2.COLOR_GRAY2BGR)
    Ci = np.linalg.inv(W[i])
    for n in names:
        for M, col in ((P[f'cam_T_{n}'][i], (0, 255, 0)), (Ci @ gtw[n][i], (0, 0, 255))):
            X = V[n] @ M[:3, :3].T + M[:3, 3]; X = X[X[:, 2] > 0.05]; uv = (X / X[:, 2:]) @ KR.T
            for u, v in uv[:, :2].astype(int):
                if 0 <= u < 1600 and 0 <= v < 1000: cv2.circle(img, (u, v), 2, col, -1)
    cv2.putText(img, f'f{i}', (20, 60), 0, 2, (255, 255, 0), 3); tiles.append(cv2.resize(img, (640, 400)))
cv2.imwrite(f'{O}/diag.jpg', np.vstack([np.hstack(tiles[:4]), np.hstack(tiles[4:])]))
print('wrote', f'{O}/diag.jpg')

In [ ]:
!python /content/diag.py 12 /content/perc 20
display(Image('/content/perc/ep12/diag.jpg', width=1000))

### 4e. Optional: all 20 public episodes (about 45 minutes)
This downloads the other 19 public episodes (black-and-white cameras + mocap) and all object scans, runs `perc.py` on each, and prints the summary table. Section 5 shows what we got.

In [ ]:
%%writefile /content/run_public.sh
#!/bin/bash
# Fetch all public Track 3 episodes (cams b/c, parquet) + all meshes, then run perc.py on each.
# Usage: setsid nohup bash /content/run_public.sh "0 2 11 ..." > /content/run_public.log 2>&1 < /dev/null &
EPS=${1:-"0 2 11 12 13 18 21 22 23 24 26 27 31 39 40 41 42 43 44 45"}
B=https://huggingface.co/datasets/nvidia/video_to_data_challenge/resolve/main/track_3/public
mkdir -p /content/t3/v /content/perc
for o in beige_cup blue_cup mini_dust_pan mini_sweeper plastic_dish_rack water_pitcher white_pot white_pot_lid wooden_piece_1 wooden_piece_2; do
  mkdir -p /content/t3/mesh/$o
  for g in $o ${o}_visual ${o}_collision; do f=/content/t3/mesh/$o/$g.glb; [ -s $f ] || curl -fsSL -o $f $B/mesh/$o/$g.glb || rm -f $f; done
done
for EP in $EPS; do
  for c in b c; do f=/content/t3/v/ep${EP}_$c.mp4; [ -s $f ] || curl -fsSL -o $f $B/videos/chunk-000/observation.images.ego_cam_$c/episode_$(printf %06d $EP).mp4; done
  f=/content/t3/ep${EP}.parquet; [ -s $f ] || curl -fsSL -o $f $B/data/chunk-000/episode_$(printf %06d $EP).parquet
done
echo FETCH-OK
for EP in $EPS; do
  mkdir -p /content/perc/ep$EP
  EP=$EP python /content/perc.py > /content/perc/ep$EP/log.txt 2>&1 && echo "EP $EP OK" || echo "EP $EP FAIL"
done
echo RUN-DONE

In [ ]:
%%writefile /content/summ.py
import json, glob, sys, numpy as np
rows = []
for f in sorted(glob.glob((sys.argv[1] if len(sys.argv) > 1 else '/content/perc') + '/ep*/result.json'), key=lambda p: int(p.split('/ep')[1].split('/')[0])):
    r = json.load(open(f)); a = [k for k in r if k.startswith('anchor:')]
    best = max(a, key=lambda k: r[k]['AUC'])
    rows.append((r['episode'], r['vo']['AUC'], r['vo']['SP_SR'], r['vo']['MP_SR'], r['vo']['MPPE_cm'], r[best]['AUC'], r['vo_stats']['keyframes'],
                 ' '.join(f"{n[:10]}:{x:.2f}" for n, x in r['vo']['per_obj_AUC'].items())))
for x in rows: print(f'{x[0]:3d} vo {x[1]:.3f} SP {x[2]:.0f} MP {x[3]:.0f} MPPE {x[4]:5.2f} | bestanchor {x[5]:.3f} kf {x[6]:3d} | {x[7]}')
v = np.array([x[1:6] for x in rows], float)
print('MEAN n=%d  vo AUC %.3f SP %.2f MP %.2f MPPE %.2f | best-anchor AUC %.3f (oracle)' % (len(rows), *v.mean(0)))

In [ ]:
!bash /content/run_public.sh > /content/run_public.log 2>&1; grep EP /content/run_public.log | tr '\n' ' '
!python /content/summ.py /content/perc

## 5. Results on the 20 public episodes (perception only, camera from visual odometry)

| ep | objects | AUC | note |
|---|---|---|---|
| 12 | blue cup, dish rack | **0.995** | |
| 13 | blue cup, dish rack | **0.998** | |
| 23 | wooden piece 1, 2 | **0.995** | |
| 27 | dish rack, white pot | **0.994** | |
| 40 | dustpan | **0.986** | |
| 22 | blue cup, water pitcher | 0.962 | |
| 18 | hand broom, dustpan | 0.935 | |
| 44 | white pot | 0.887 | |
| 31 | hand broom, dish rack | 0.885 | |
| 11 | blue cup, beige cup | 0.871 | twin cups; the static cup drifts with camera tracking |
| 21 | blue cup, water pitcher | 0.858 | |
| 45 / 42 / 43 / 41 | white pot | 0.836 / 0.817 / 0.807 / 0.795 | long camera-tracking chains, so drift |
| 24 | wooden piece 1, 2 | 0.820 | |
| 39 | dustpan | 0.550 | about a 5 cm offset once it moves |
| 0 | white pot, pot lid | 0.179 | lid is 180°-symmetric, so its rotation is wrong |
| 26 | dish rack, white pot | 0.031 | white pot inside a white rack, so detection fails |
| 2 | white pot, pot lid | ≈0.98 | not read individually; inferred from the mean |
| **mean (20)** | | **0.809** | SP-SR 0.90, MP-SR 0.85, MPPE 4.8 cm |

**What this means:**
- When an object is found correctly, tracking is **almost perfect**: about 1 cm error even on fast motions.
- Our mean **from video alone** (0.809) is above the leader's **final** score (0.765). That is a strong sign, but **not yet a leaderboard result**: the robot still has to reproduce the motion in simulation, and that always loses some accuracy.
- The failures are specific and fixable:
  - **Symmetric lid:** use the small mocap markers on the lid to tell its rotation. Our earlier research did this with a marker-only lid tracker, which scored AUC 0.977.
  - **White-on-white pot and rack:** a render-and-compare check (built, being tested).
  - **Camera drift on long pot videos:** improved camera tracking (built, being tested).

**Lessons from the way there:**
- **Text prompts don't work on these cameras.** A text-prompted detector (OWLv2, e.g. "a cup") misses the pot on black-and-white images and swapped the two cups (AUC 0.10 → 0.87 after the switch to scan-based matching).
- **FoundationPose's internal score cannot compare different regions.** A piece of the rack outscored the real cup, 143 vs 73. So we pick regions with our own checks, not with that score.

## 6. Hands (step 2): first test
The robot needs to know where the person's hands and fingers are. Several popular hand models need the licensed **MANO** hand model, so we use Google's **MediaPipe Hands** instead, which is free. We then lift its 21 points per hand to 3-D with our stereo depth.

**What we saw.** MediaPipe almost never finds hands in the black-and-white cameras: 0 two-hand frames on episodes 0 and 12. On the **colour** camera it finds both hands in **173 of 181** frames (episode 12) and **338 of 373** (episode 0). So hands come from the colour camera, projected into the stereo depth through the factory calibration.

The cell below repeats the detection count. It needs a small extra environment with MediaPipe 0.10.14.

In [ ]:
!uv venv -q --python 3.11 /content/mpvenv && uv pip install -q --python /content/mpvenv/bin/python 'mediapipe==0.10.14' opencv-python-headless numpy
!for c in a c; do f=/content/t3/v/ep12_$c.mp4; [ -s $f ] || curl -fsSL -o $f https://huggingface.co/datasets/nvidia/video_to_data_challenge/resolve/main/track_3/public/videos/chunk-000/observation.images.ego_cam_$c/episode_000012.mp4; done

In [ ]:
%%writefile /content/mptest.py
import sys, numpy as np, cv2, mediapipe as mp
EP, CAM = int(sys.argv[1]), sys.argv[2]
H = mp.solutions.hands.Hands(static_image_mode=False, max_num_hands=2, model_complexity=1, min_detection_confidence=0.3, min_tracking_confidence=0.3)
cap = cv2.VideoCapture(f'/content/t3/v/ep{EP}_{CAM}.mp4'); cnt = []
while True:
    ok, f = cap.read()
    if not ok: break
    cnt.append(len(H.process(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)).multi_hand_landmarks or []))
cnt = np.array(cnt); print(f'episode {EP} camera {CAM}: frames with 0/1/2 hands =', [int((cnt == k).sum()) for k in range(3)])

In [ ]:
!/content/mpvenv/bin/python /content/mptest.py 12 a 2>/dev/null; /content/mpvenv/bin/python /content/mptest.py 12 c 2>/dev/null

## 7. What's next, and what it costs
1. **Reference + robot training on public episodes.** We turn our object and hand tracks into the simulator's target format, train a policy, run the official evaluator, and score against mocap. **This is the decisive test**: it tells us how much accuracy the robot stage loses. It takes about 2-3 GPU hours.
2. **The wooden spoon** is in 10 of the 20 eval episodes and has **no scan**. We built a parametric spoon model (about 30 cm, 95×52 mm bowl, three mocap markers). Its exact coordinate convention is close to a coin flip, so we will ask the organisers.
3. **Pot and lid** (6 eval episodes): lid symmetry via markers, plus the fixes above.
4. **First submission:** 20 eval policies at about 2 h each, so about 40 GPU hours. On a rented RTX PRO 6000 (about $2/h) the first full leaderboard submission costs roughly **$150-200**.

**Hardware notes:**
- The full training recipe needs a GPU with **≥ 48 GB** of memory: we measured 40.5 GB.
- Perception needs much less.
- Everything runs on Linux, with an NVIDIA driver ≥ 580 (CUDA 13).

**Deadline:** leaderboard freeze is 4 Nov 2026.

---
### Glossary
- **6-D pose**: an object's position (x, y, z) plus rotation (3 angles).
- **Stereo / disparity**: two cameras side by side. How far a point shifts between the two images (disparity) gives its depth.
- **Rectification**: warping the stereo images so matching points lie on the same row.
- **Mocap**: motion capture. Small reflective markers on the objects are tracked by external cameras; this is the "true" motion.
- **Visual odometry (VO)**: estimating how the camera moved from the images themselves.
- **Policy**: the neural-network controller that drives the robot hands. **RL** (reinforcement learning) trains it by trial and error in simulation.
- **Reference**: the target motion (objects + hands) the policy is trained to reproduce.
- **AUC / SP-SR / MP-SR / MPPE**: the leaderboard metrics; see section 0.